# Bronze Layer — Raw Data Ingestion

Fetches 5 years of daily weather data from the Open-Meteo Archive API for 36 Indian states and union territories.

| Column | Type | Description |
|--------|------|-------------|
| `DATE` | `datetime64` | Observation date |
| `CITY` | `str` | State or union territory name |
| `TEMPERATURE_C` | `float64` | Daily mean temperature |
| `PRECIPITATION_MM` | `float64` | Daily precipitation sum |
| `WIND_SPEED_KMH` | `float64` | Daily max wind speed |

In [ ]:
import time
from datetime import date, datetime, timedelta

import pandas as pd
import requests
import yaml

ARCHIVE_URL   = "https://archive-api.open-meteo.com/v1/archive"
DAILY_VARS    = ["temperature_2m_mean", "precipitation_sum", "wind_speed_10m_max"]
YEARS         = 5
CHUNK_SIZE    = 10

In [ ]:
with open("/Workspace/scripts/india/configuration.yaml") as f:
    cfg = yaml.safe_load(f)

regions    = cfg["regions"]
end_date   = date.today() - timedelta(days=1)
start_date = end_date.replace(year=end_date.year - YEARS)

print(f"Date range : {start_date} -> {end_date}  ({YEARS} years)")
print(f"Regions    : {len(regions)}")

## Regions & Coordinates

In [ ]:
coords = {r["name"]: (r["lat"], r["lon"]) for r in regions}
for region, (lat, lon) in coords.items():
    print(f"  {region:<42}  lat={lat:>9.4f}  lon={lon:>10.4f}")

## Fetch Weather Data

In [ ]:
def _get_with_retry(url: str, params: dict, timeout: int = 30) -> requests.Response:
    """
    GET with retry on HTTP 429 / 5xx: exponential backoff of 30, 60, 120, 240
    seconds (4 retries). Re-raises if all attempts fail. Other 4xx errors are
    not retried.
    """
    backoffs = [30, 60, 120, 240]
    for i in range(len(backoffs) + 1):
        resp = requests.get(url, params=params, timeout=timeout)
        if resp.status_code == 429 or resp.status_code >= 500:
            if i == len(backoffs):
                resp.raise_for_status()
            wait = backoffs[i]
            print(f"  rate limited ({resp.status_code}) — retrying in {wait}s "
                  f"(attempt {i + 1}/{len(backoffs)})")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        return resp


def fetch_weather_batch(region_chunk):
    """
    Fetch daily weather for multiple regions in a single Open-Meteo request
    (comma-separated latitude/longitude). Returns a flat list of
    (datetime, region, temperature_c, precipitation_mm, wind_speed_kmh) tuples,
    covering all regions in region_chunk, in region order.
    """
    lats = ",".join(str(coords[r][0]) for r in region_chunk)
    lons = ",".join(str(coords[r][1]) for r in region_chunk)

    resp = _get_with_retry(
        ARCHIVE_URL,
        params={
            "latitude": lats, "longitude": lons,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "daily": ",".join(DAILY_VARS),
            "timezone": "auto",
        },
    )
    payload = resp.json()
    results = payload if isinstance(payload, list) else [payload]

    if len(results) != len(region_chunk):
        raise RuntimeError(
            f"Open-Meteo returned {len(results)} result(s) for {len(region_chunk)} "
            "requested region(s) — refusing to zip by index, since a mismatch "
            "would misattribute weather data to the wrong region."
        )

    chunk_rows = []
    for region, data in zip(region_chunk, results):
        daily = data["daily"]
        rows = []
        for t, te, pr, wi in zip(daily["time"], daily["temperature_2m_mean"],
                                  daily["precipitation_sum"], daily["wind_speed_10m_max"]):
            if te is None or wi is None:
                continue
            pr = pr if pr is not None else 0.0
            rows.append((datetime.strptime(t, "%Y-%m-%d"), region, float(te), float(pr), float(wi)))
        chunk_rows.append((region, rows))
    return chunk_rows


all_rows = []
region_names = list(coords.keys())
for i in range(0, len(region_names), CHUNK_SIZE):
    chunk = region_names[i:i + CHUNK_SIZE]
    chunk_results = fetch_weather_batch(chunk)

    for region, rows in chunk_results:
        all_rows.extend(rows)
        if rows:
            print(f"  {region:<42}  {len(rows):>5} rows  ({rows[0][0].date()} -> {rows[-1][0].date()})")
        else:
            print(f"  {region:<42}  {0:>5} rows  (no data)")

    if i + CHUNK_SIZE < len(region_names):
        time.sleep(2)

print(f"\nTotal rows fetched: {len(all_rows):,}")

## Assemble Bronze DataFrame

In [ ]:
bronze_df = pd.DataFrame(all_rows, columns=[
    "DATE", "CITY", "TEMPERATURE_C", "PRECIPITATION_MM", "WIND_SPEED_KMH",
])
bronze_df["DATE"] = pd.to_datetime(bronze_df["DATE"])
for col in ["TEMPERATURE_C", "PRECIPITATION_MM", "WIND_SPEED_KMH"]:
    bronze_df[col] = bronze_df[col].astype("float64")

# -- Spark conversion (commented out for future cluster deployment) ----------
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
bronze_sdf = spark.createDataFrame(bronze_df)

print(f"bronze_df: {bronze_df.shape}")
bronze_df.head()

In [ ]:
# Create bronze catalog

bronze_catalog = "bronze_weather_india"
bronze_schema = "weather"
bronze_table = "raw_weather"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {bronze_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_catalog}.{bronze_schema}")

bronze_sdf.write \
	.mode("overwrite") \
	.format("csv") \
    .option("header", "true") \
	.option("sep", ",") \
	.option("quote", '"') \
	.option("escape", "\\") \
	.saveAsTable(f"{bronze_catalog}.{bronze_schema}.{bronze_table}")

In [ ]:
df = spark.read.table("bronze_weather_india.weather.raw_weather")
df.show()